# 01 — PHANOTATE Phage Genome Annotation

This notebook annotates phage genomes using PHANOTATE and produces INTERFACE-compliant outputs.

**Reference:** McNair K. et al. (2019) *PHANOTATE: a novel approach to gene identification in phage genomes.* Bioinformatics 35(22):4537–4542. DOI: 10.1093/bioinformatics/btz265

**Why PHANOTATE?** Phage genomes contain overlapping ORFs (~10% of genes overlap); PHANOTATE uses a graph/Bellman-Ford model that tolerates and correctly calls overlapping CDSs, outperforming Prodigal by ~15% on phage-specific benchmarks (McNair 2019). Prodigal assumes non-overlapping ORFs and is NOT used for phages.

---

## 中文说明

本 Notebook 使用 PHANOTATE 对噬菌体基因组进行注释，输出符合 INTERFACE.md 规范的文件。

**为何选择 PHANOTATE？** 噬菌体基因组中约 10% 的基因相互重叠；PHANOTATE 采用图算法（Bellman-Ford 最短路径）可正确识别重叠 ORF，在噬菌体基准测试中比 Prodigal 多检出约 15% 的基因（McNair 2019）。Prodigal 假设 ORF 不重叠，**不得用于噬菌体**。

In [ ]:
# Cell 2 — Imports and version printouts
# 导入库并打印版本信息
import sys
import subprocess
from pathlib import Path

# Path anchoring: notebooks live in <module>/processes/, so parents[1] = module root
# 路径锚定：notebook 在 <module>/processes/ 中，parents[1] = 模块根目录
NOTEBOOK_DIR = Path.cwd().resolve()
MODULE_ROOT  = NOTEBOOK_DIR.parent          # 02_annotation/
REPO_ROOT    = MODULE_ROOT.parent            # iGEM_Claremont_2026/
sys.path.insert(0, str(NOTEBOOK_DIR))

from annotate_lib import PHANOTATE_VERSION, run_phanotate, append_manifest, manifest_row_for_file
from Bio import SeqIO
import Bio

print(f"Python:         {sys.version}")
print(f"Biopython:      {Bio.__version__}")
print(f"PHANOTATE:      {PHANOTATE_VERSION}")
print(f"REPO_ROOT:      {REPO_ROOT}")
print(f"MODULE_ROOT:    {MODULE_ROOT}")

## Method: PHANOTATE algorithm / 方法说明

PHANOTATE encodes all possible ORFs in a directed graph. Each edge weight encodes the probability that a given ORF is *not* a functional gene (based on codon usage statistics trained on known phage proteomes). The optimal annotation is the **minimum-weight path** through the graph (Bellman-Ford). Unlike Prodigal, this formulation allows overlapping ORFs on the path.

PHANOTATE 将所有可能的 ORF 编码为有向图。每条边的权重代表该 ORF 不是功能基因的概率（基于已知噬菌体蛋白质组的密码子使用统计）。最优注释是图中的**最小权重路径**（Bellman-Ford 算法）。与 Prodigal 不同，该方法允许路径上的 ORF 相互重叠。

**Output files / 输出文件:**
- `outputs/phage_proteins/<acc>.faa` — protein FASTA with INTERFACE headers
- `outputs/phage_orfs/<acc>.gff3`  — ORF coordinates in GFF3 format

In [ ]:
# Cell 4 — Helper functions (defined in annotate_lib.py)
# 辅助函数在 annotate_lib.py 中定义，此处直接调用
# run_phanotate(genome_fna, output_dir) -> dict
#   Wraps PHANOTATE CLI, parses tabular output, translates to protein FASTA + GFF3
#   封装 PHANOTATE CLI，解析表格输出，翻译为蛋白质 FASTA + GFF3

OUTDIR = MODULE_ROOT / "outputs"
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTDIR}")

## Sample run: phiL7 (EU717894.1)

Tonight's sample run annotates **phiL7** only. Full batch over all 777 phages is deferred to the Laguna HPC run (see LAGUNA.md).

今晚的样本运行**仅注释 phiL7**。对全部 777 个噬菌体的批量注释推迟到 Laguna HPC 运行（见 LAGUNA.md）。

In [ ]:
# Cell 8 — Sample run on phiL7 / 对 phiL7 的样本运行
PHAGE_FNA = REPO_ROOT / "00_raw_data" / "phage" / "EU717894.1" / "genome.fna"
print(f"Input: {PHAGE_FNA}")
assert PHAGE_FNA.exists(), f"genome.fna not found: {PHAGE_FNA}"

meta = run_phanotate(PHAGE_FNA, OUTDIR)
print(f"\nResults for {meta['acc']}:")
print(f"  ORFs:         {meta['n_orfs']}")
print(f"  Mean len:     {meta['mean_orf_len']} aa")
print(f"  Runtime:      {meta['runtime_s']} s")
print(f"  FAA:          {meta['faa_path']}")
print(f"  GFF3:         {meta['gff3_path']}")

In [ ]:
# Cell 9 — Validation: check FASTA header format and ORF count
# 验证：检查 FASTA 头格式和 ORF 数量
import re

HEADER_RE = re.compile(
    r'^>(\S+) \| source=(\S+) \| length=(\d+)'
    r' \| start=(\d+) \| end=(\d+) \| strand=([+-])'
    r' \| tool=(\S+)$'
)

faa_path = Path(meta['faa_path'])
headers = [l for l in faa_path.read_text().splitlines() if l.startswith('>')]

bad = [h for h in headers if not HEADER_RE.match(h)]
assert not bad, f"Malformed headers: {bad[:3]}"
print(f"All {len(headers)} headers pass INTERFACE regex ✓")

# Lee 2009 reports 59 ORFs; PHANOTATE 1.6.7 finds 80 (includes smaller overlapping ORFs)
# Lee 2009 报告 59 个 ORF；PHANOTATE 1.6.7 找到 80 个（包含更多小重叠 ORF）
assert 50 <= len(headers) <= 90, f"ORF count {len(headers)} outside expected range [50, 90]"
print(f"ORF count {len(headers)} in expected range [50, 90] ✓")

In [ ]:
# Cell 10 — Optional batch run (skip tonight — deferred to Laguna)
# 可选批量运行（今晚跳过 — 推迟到 Laguna）
# To run all phages: uncomment and execute on Laguna HPC
# 如需运行全部噬菌体：取消注释并在 Laguna HPC 上执行

BATCH_ENABLED = False   # set True on Laguna / 在 Laguna 上设为 True

if BATCH_ENABLED:
    from concurrent.futures import ProcessPoolExecutor, as_completed
    phage_dirs = sorted((REPO_ROOT / "00_raw_data" / "phage").iterdir())
    batch_results = []
    with ProcessPoolExecutor(max_workers=8) as pool:
        futures = {
            pool.submit(run_phanotate, d / "genome.fna", OUTDIR): d.name
            for d in phage_dirs if (d / "genome.fna").exists()
        }
        for fut in as_completed(futures):
            try:
                batch_results.append(fut.result())
            except Exception as exc:
                print(f"FAILED {futures[fut]}: {exc}")
    print(f"Batch complete: {len(batch_results)} phages annotated")
else:
    print("Batch run skipped (BATCH_ENABLED=False). Set True for Laguna HPC run.")

In [ ]:
# Cell 11 — Append to MANIFEST.csv
# 追加到 MANIFEST.csv
MANIFEST = OUTDIR / "MANIFEST.csv"
rows = [
    manifest_row_for_file(meta['faa_path'],  meta['n_orfs'], "PHANOTATE protein FASTA"),
    manifest_row_for_file(meta['gff3_path'], meta['n_orfs'], "PHANOTATE GFF3 ORF coords"),
]

# Only append if not already recorded (idempotent)
# 仅在未记录时追加（幂等操作）
existing_files = set()
if MANIFEST.exists():
    import csv
    with open(MANIFEST) as f:
        existing_files = {r['filename'] for r in csv.DictReader(f)}

new_rows = [r for r in rows if r['filename'] not in existing_files]
if new_rows:
    append_manifest(MANIFEST, new_rows)
    print(f"Added {len(new_rows)} rows to MANIFEST.csv")
else:
    print("MANIFEST.csv already up to date.")